# **LiminalGPT Stage 2**: _Self attention_ & _Residual Connections_

A basic transformer based neural network trained on a literary text corpus.

In Stage 2, we explore the concept of self attention in a transformer architecture.

LiminalGPT is based on [Vaswani et al. (2017)](https://arxiv.org/pdf/1706.03762).

Notebook content inspired by Andrej Kaparthy's [GPT lecture](https://www.youtube.com/watch?v=kCc8FmEb1nY).


In [111]:
import torch
from typing import Final

SEED: Final[int] = 3433

torch.manual_seed(SEED);

## Building the intuition behind self attention

We will compute a running average of feature vectors for each token position. For a token at position `i`, we average the features from all tokens at positions `0` through `i` (inclusive). This creates a representation that captures the "context" or "history" leading up to each token.

Given a tensor of shape `(B, T, C)` where:

- `B` = batch size
- `T` = sequence length (time steps)
- `C` = number of channels (feature dimensions)

For each token, we want to aggregate information only from previous tokens and itself (not future tokens).


### Version 1: weighted aggregation via manual loops


In [112]:
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)
x.shape

torch.Size([4, 8, 2])

For now, let us perform the weighted aggregation for each token via loops:


In [113]:
xbow = torch.zeros((B, T, C))

for b in range(B):
    for t in range(T):
        # extract running history of tokens
        xprev = x[b, : t + 1]  # shape (T, C)
        # calculate running mean
        xbow[b, t] = xprev.mean(0)

In [114]:
x[0]

tensor([[-1.4970, -0.6357],
        [-0.6095, -0.1586],
        [ 0.9341, -0.0361],
        [-0.4688, -0.0288],
        [-1.2407,  0.2724],
        [ 0.1070, -1.9325],
        [ 0.8531, -0.5161],
        [ 0.0290, -0.1809]])

In [115]:
xbow[0]

tensor([[-1.4970, -0.6357],
        [-1.0532, -0.3972],
        [-0.3908, -0.2768],
        [-0.4103, -0.2148],
        [-0.5764, -0.1174],
        [-0.4625, -0.4199],
        [-0.2745, -0.4336],
        [-0.2366, -0.4020]])

### Version 2: weighted aggregation via matrix multiplication


#### 2.a Calculating global mean


In [116]:
a = torch.ones(3, 3)
print(f"{a = }")

b = torch.randint(0, 10, (3, 2)).float()
print(f"\n{b = }")

c = a @ b
print(f"\n{c = }")

mean = c.mean(0)
print(f"\n{mean = }")

a = tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])

b = tensor([[1., 4.],
        [4., 9.],
        [0., 4.]])

c = tensor([[ 5., 17.],
        [ 5., 17.],
        [ 5., 17.]])

mean = tensor([ 5., 17.])


#### 2.b Calculating running mean via `torch.tril()`


[torch.tril()](https://docs.pytorch.org/docs/stable/generated/torch.tril.html) converts all elements above the diagonal to 0:


In [117]:
ex = torch.ones(3, 3)
print(f"before tril():\n{ex}")
print("\n")
ex = torch.tril(ex)
print(f"after tril():\n{ex}")

before tril():
tensor([[1., 1., 1.],
        [1., 1., 1.],
        [1., 1., 1.]])


after tril():
tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])


We can use `tril()` in matrix multiplication to calculate the running mean:


In [118]:
a = torch.tril(torch.ones(3, 3))
print(f"{a = }")

b = torch.randint(0, 10, (3, 2)).float()
print(f"\n{b = }")

c = a @ b
print(f"\n{c = }")

a = tensor([[1., 0., 0.],
        [1., 1., 0.],
        [1., 1., 1.]])

b = tensor([[9., 1.],
        [5., 9.],
        [8., 4.]])

c = tensor([[ 9.,  1.],
        [14., 10.],
        [22., 14.]])


But how do we calculate the mean from the running sums?

We can perform a mean by **normalizing the rows** of tensor `a`:


In [119]:
a = a / a.sum(1, True)
a

tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

We know:

$$mean = \frac{1}{\text{total elements}}\times\text{sum of elements} $$

For a running average, we want each row of `a` to be weights that sum to 1, so the result is the average and not the sum.

In the example above:

Row 0: [1.0, 0, 0] → sums to 1 → averages token 0 only

Row 1: [0.5, 0.5, 0] → sums to 1 → averages tokens 0-1

Row 2: [0.33, 0.33, 0.33] → sums to 1 → averages tokens 0-2

Now when we do `a @ b`, we get the running mean instead of the running sum:

- Position 0:
  $$1.0 × b[0] = \textcolor{lightblue}{\frac{1}{1} \times (b[0])} =  \text{just b[0]}$$

- Position 1:
  $$0.5 × b[0] + 0.5 × b[1] = \textcolor{lightblue}{\frac{1}{2} \times (b[0] + b[1])} = \text{mean of } b[0] \text{ and } b[1]$$

- Position 2:
  $$0.33 × b[0] + 0.33 × b[1] + 0.33 × b[2] = \textcolor{lightblue}{\frac{1}{3} \times (b[0] + b[1] + b[2])} = \text{mean of all three}$$

Observe how these $\textcolor{lightblue}{\text{expressions}}$ match our _mean_ formula!

Therefore, if we normalize the weights so that they sum to 1, we transform the matrix multiplication from computing a weighted sum into computing a **weighted average**. This is exactly what our manual loop implementation was trying to do!


#### 2.c Complete vectorized method


In [120]:
# version 1
B, T, C = 4, 8, 2
x = torch.randn(B, T, C)

xbow = torch.zeros((B, T, C))

for b in range(B):
    for t in range(T):
        # extract running history of tokens
        xprev = x[b, : t + 1]  # shape (T, C)
        # calculate running mean
        xbow[b, t] = xprev.mean(0)

In [121]:
# version 2
weights = torch.tril(torch.ones(T, T))  # reason for shape explained below
weights = weights / weights.sum(1, True)
weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [122]:
#  (T, T) @ (B,T,C) --> (B, T, T) @ (B, T, C) --> (B,T,C)
# we want the output to have same shape as x so weights shape must be (T,T)
xbow2 = weights @ x
print("xbow and xbow2 have close values:", torch.allclose(xbow, xbow2))
# compare batch 1
print("\nfirst batch of xbow and xbow2:")
xbow[0], xbow2[0]

xbow and xbow2 have close values: True

first batch of xbow and xbow2:


(tensor([[ 0.2356,  0.1581],
         [ 0.6854, -0.1624],
         [ 0.5331, -0.1489],
         [ 0.5530,  0.1019],
         [ 0.6846, -0.0534],
         [ 0.5232,  0.0711],
         [ 0.5349, -0.0523],
         [ 0.5083,  0.0794]]),
 tensor([[ 0.2356,  0.1581],
         [ 0.6854, -0.1624],
         [ 0.5331, -0.1489],
         [ 0.5530,  0.1019],
         [ 0.6846, -0.0534],
         [ 0.5232,  0.0711],
         [ 0.5349, -0.0523],
         [ 0.5083,  0.0794]]))

### Version 3: weighted aggregation via softmax


Read about [torch.Tensor.masked_fill()](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.masked_fill_.html).

This version achieves the same aggregation effect but helps us reason about the mechanism in a different pov.

---

In all examples, we intitialized weight matrix to zero to demonstrate the mechanics of aggregation.

In practice, these weights are data-dependent and learned from the input.
We can think of their magnitude as interaction strength or affinity between tokens:

- Higher weights mean stronger connections between token pairs
- Lower weights mean weaker connections

Let sequence: ["The", "cat", "sat"]

For this sequence we would have a 3x3 weight matrix with one entry for every possible pair of tokens in the sequence:

- w[0,0]: pair ("The" → "The") - how much "The" attends to itself
- w[1,0]: pair ("cat" → "The") - how much "cat" attends to "The"
- w[1,2]: pair ("cat" → "sat") - how much "cat" attends to "sat" (this would be masked to -inf in causal attention since "sat" comes after "cat")

The weights determine how much each token _pays attention_ to other tokens. :)

---

The masked fill operation can be thought of as a clamping operation that ensures tokens from the future cannot communicate/aggregate.

By setting future positions to -inf before softmax, we enforce causality:

- Token at position t can only attend to tokens at positions 0 through t (including itself)
- Token at position t cannot "see" tokens at positions t+1, t+2, ..., T-1

Note:

> LiminalGPT is a decoder-only transformer that uses causal masking like ChatGPT.
>
> We must prevent information leakage from future tokens. For a sequence of length T, we create T training examples during causal language modeling: predicting token 1 from token 0, predicting token 2 from tokens 0-1, and so on. Without masking, the model could "cheat" by attending to future tokens it's supposed to predict.
>
> For example: when learning to predict position i, the model could simply copy the answer from position i directly rather than learning the underlying patterns.

In contrast, encoder-decoder transformers like those used for translation have different masking behavior:

- The **encoder** uses bidirectional attention (no masking) because it processes the complete input sequence.
- The **decoder** uses causal masking for self-attention.
- Cross-attention from decoder to encoder is unmasked. The decoder can attend to all encoder outputs.

The mask ensures that the model learns to predict each token using only the context that came before it.


In [123]:
tril = torch.tril(torch.ones(T, T))

weights = torch.zeros(T, T)
weights = weights.masked_fill(tril == 0, float("-inf"))
weights

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

Softmax-ing along dim 1 gives us the same weight tensor as in version 2:

Softmax exponentiates each element and divides each element by the sum of all elements. Here:

- $\exp(-\infty) = 0$
- $\exp(0) = 1$


In [124]:
weights = weights.softmax(1)
weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])

In [125]:
# same aggregation as v2 via matrix mul
xbow3 = weights @ x

print("xbow and xbow3 have close values:", torch.allclose(xbow, xbow3))
# compare batch 1
print("\nfirst batch of xbow and xbow3:")
xbow[0], xbow3[0]

xbow and xbow3 have close values: True

first batch of xbow and xbow3:


(tensor([[ 0.2356,  0.1581],
         [ 0.6854, -0.1624],
         [ 0.5331, -0.1489],
         [ 0.5530,  0.1019],
         [ 0.6846, -0.0534],
         [ 0.5232,  0.0711],
         [ 0.5349, -0.0523],
         [ 0.5083,  0.0794]]),
 tensor([[ 0.2356,  0.1581],
         [ 0.6854, -0.1624],
         [ 0.5331, -0.1489],
         [ 0.5530,  0.1019],
         [ 0.6846, -0.0534],
         [ 0.5232,  0.0711],
         [ 0.5349, -0.0523],
         [ 0.5083,  0.0794]]))

## Building self attention


### Keys, Values and Queries

In self-attention, keys `k`, queries `q`, and values `v` (all shaped B, T, head_size) are three different vector representations of a token. Keys and queries work together to compute the attention weights, which encapsulate token affinity. Values contain the actual information that gets aggregated.

The dimension of these vectors is determined by hyperparameter `head_size`:

$$head\_size = \frac{\text{n\textunderscore embd}}{\text{num\textunderscore heads}}$$

- **Query**: A vector that represents what information a token wants to find from other tokens in a sequence.

- **Key**: A vector that represents what information a token has to offer.

- **Value**: A vector that represents the actual information content that will be passed forward when this token is attended to.


Remember that our input tokens start with embeddings in `C`-dimensional space. These embeddings capture general information about each token.

But for the concept of "attention", we need a mechanism that quantifies the aggregated information (via a score) about _contextually-similar_ tokens. Our current embeddings do not possess that capability by itself.

Let us transform our token embeddings into three separate **learnable** "semantic spaces" where such comparisons can happen:

- Weight $W_{key}$ projects the `C`-embeddings into a _key space_ while emphasizing the characteristics that make this token useful to others and learning to extract the features that it offers to other tokens.

- Weight $W_{query}$ projects the `C`-embeddings into a _query space_ while emphasizing the requirements of this token and learning to extract the features that it is searching for.

- Weight $W_{value}$ projects the `C`-embeddings into a _value space_ while learning to extract and transform the actual information that will be communicated and aggregated. The value is what gets passed along when attention is paid to this token.

If we used the same transformation for both keys and queries (or no transformation at all), every token would be asking for exactly what it offers, which limits the model's ability to learn complex relationships.

So, we have three separate learned matrices for each token to ensure that:

- A token can need something different from what it provides
- The model learns asymmetric relationships
- What gets communicated (value) can be different from both what's offered (key) and what's requested (query)

Consider this example:

token = "cat" and embedding = e_cat

```py
k_cat = e_cat @ W_key    # "I represent an animal/subject"
q_cat = e_cat @ W_query  # "I'm looking for actions that involve me"
```

token = "sat" and embedding = e_sat

```py
k_sat = e_sat @ W_key    # "I represent a past-tense action"
q_sat = e_sat @ W_query  # "I'm looking for the subject doing this action"
```

When we compute `q_sat · k_cat` (query of "sat" with key of "cat"), we get a high score because:

- "sat" is asking for a subject

- "cat" is offering subject information

The weight matrices learn these transformations during training to maximize the model's ability to predict the next token correctly.


### Single _head_ performing self-attention


#### Token embeddings $\rightarrow$ keys, values and queries


In [126]:
import torch.nn as nn

B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# number of dims for key and query vector
head_size = 16

# linear layers that transform token embeddings into key, value and query spaces
key = nn.Linear(C, head_size, bias=False)  # x @ W_key
value = nn.Linear(C, head_size, bias=False)  # x @ W_value
query = nn.Linear(C, head_size, bias=False)  # x @ W_query
# (B,T,C) @ (C, 16) --> (B, T, 16)
k, q, v = key(x), query(x), value(x)

#### "Attention Pattern" matrix


In [ ]:
# (B, T, 16) @ (B, 16, T) --> (B, T, T)
weights = q @ k.transpose(-2, -1)
# scale by 1/sqrt(head_size) to prevent large dot products (explanation below)
weights = weights * (head_size**-0.5)

# mask for causal attention (only in decoder-style transformers)
# masking is ignored in encoder-style transformers
mask = torch.tril(torch.ones(T, T))
# make upper triangle of weights all -inf (so tokens cant see future)
weights = weights.masked_fill(mask == 0, float("-inf"))
# row normalization:
# upper diagonal of weights also all zeros from softmax
# each row sums to 1
attention_matrix = weights.softmax(-1)

In [128]:
attention_matrix.shape, attention_matrix[0]

(torch.Size([4, 8, 8]),
 tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4966, 0.5034, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2382, 0.3453, 0.4165, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1928, 0.1657, 0.2893, 0.3523, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1173, 0.2179, 0.2916, 0.1781, 0.1951, 0.0000, 0.0000, 0.0000],
         [0.1918, 0.1027, 0.1478, 0.2333, 0.1200, 0.2043, 0.0000, 0.0000],
         [0.0908, 0.1242, 0.2568, 0.0873, 0.1602, 0.1505, 0.1301, 0.0000],
         [0.1248, 0.1850, 0.1084, 0.1422, 0.1071, 0.0875, 0.1336, 0.1114]],
        grad_fn=<SelectBackward0>))

#### Weighted aggregation


In [129]:
# matrix mul weight aggregation
# attention weights determine HOW MUCH of each token's value to aggregate
# (B,T,T) @ (B,T,head_size) --> (B,T,head_size)
weighted_mean = attention_matrix @ v
weighted_mean.shape, weighted_mean[0]

(torch.Size([4, 8, 16]),
 tensor([[ 0.1160,  0.5391, -0.2171,  0.6494,  0.2697, -0.3072, -0.5532, -0.8535,
           0.0797,  0.2640,  0.0433, -1.0744, -0.2112, -1.2217, -0.3322, -1.3432],
         [-0.0687,  0.5656,  0.1870,  0.1272, -0.0200, -0.2676, -0.2644, -0.5728,
           0.2169, -0.1084, -0.3260, -0.2358, -0.1340, -0.6434,  0.3805, -0.1839],
         [ 0.1008,  0.4430,  0.1733,  0.2858, -0.1104,  0.1098,  0.1205, -0.0649,
           0.0103, -0.1890, -0.0338,  0.1557, -0.3463, -0.1045,  0.1871,  0.2394],
         [ 0.1037,  0.2014,  0.1930,  0.5404,  0.2193,  0.0905,  0.4305, -0.0928,
          -0.0329, -0.3089,  0.4860, -0.3578, -0.2323, -0.0393,  0.3666, -0.1047],
         [ 0.0893,  0.0593,  0.1407,  0.2339,  0.0269,  0.0948,  0.3686,  0.0167,
          -0.0833, -0.2994,  0.2392,  0.0752, -0.1070, -0.1360,  0.3700,  0.0655],
         [-0.0375,  0.0263,  0.1104,  0.3310,  0.3758, -0.2454,  0.2751, -0.2245,
          -0.1000, -0.2829,  0.1327, -0.1043, -0.0808, -0.2927,  0.3

Notes:

- Attention is a **communication mechanism**. Can be seen as nodes in a directed graph looking at each other and aggregating information with a weighted sum from all nodes that point to them, with data-dependent weights.

- There is no notion of space. Attention simply acts over a set of vectors. This is why we need to positionally encode tokens.

- Each example across batch dimension is processed completely independently and never "talk" to each other.

- In an "encoder" attention block, just delete the single line that does masking with `tril`, allowing all tokens to communicate. This block here is called a "decoder" attention block because it has triangular masking, and is usually used in autoregressive settings, like language modeling.


#### Self vs Cross vs Scaled attention

**Self-attention** just means that the keys and values are produced from the same source as queries.

In **cross-attention**, the queries still get produced from `x`, but the keys and values come from some other, external source (e.g. an encoder module).

**Scaled-attention** divides `weights` by 1/sqrt(head_size). This makes it so when input `q`,`k` are unit variance, `weights` will be unit variance too and softmax will stay diffuse and not saturate too much. This is illustrated below:


In [ ]:
torch.manual_seed(6356)
x = torch.randn(B, T, C)

# nn.Linear uses Kaiming Init to give outputs with var 1
# so outputs roughly follow this distribution:
k, q = torch.randn(B, T, head_size), torch.randn(B, T, head_size)

w = q @ k.transpose(-2, -1)  # (B,T,16) @ (B, 16, T) --> (B,T,T)

print("var(k)=", k.var().item())
print("var(q)=", q.var().item())
print("before normalization: var(w)=", w.var().item())

w = w * head_size**-0.5
print("after normalization: var(w)=", w.var().item())

var(k)= 0.9959767460823059
var(q)= 1.0715069770812988
before normalization: var(w)= 15.363853454589844
after normalization: var(w)= 0.9602408409118652


Notice how `w` variance scales by `head_size` where `w` variance $\approx$ `head_size`.

The intuition behind the $\frac{1}{\sqrt{\text{head\_size}}}$ scaling for $q @ k^T$ is similar to Kaiming Init:

- Each element in $q$ and $k$ has variance $\approx 1$

- Each element in `w` is derived from dot product of the sums of `head_size` terms: $(q_1k_1 + q_2k_2 + \ldots + q_nk_n)$

- Use property of independent random variables: $\text{Var}(\sum) = \sum \text{Var}$

- So the output variance $\approx$ `head_size`

- Divide `w` by $\sqrt{\text{head\_size}}$ and get:

$$\text{Var}(w/\sqrt{n}) = (\frac{1}{\sqrt{n}})^2\times\text{Var}(w) \approx \text{head\_size}/\text{head\_size} = 1$$

The scaling is required to stop saturation from the $e^x$ operation of softmax. Observe the effect of this operation on values not close to 0:

<img src="./images/softmax-graph.png" width="60%" alt="y=e^x vs y = loge(x) graph">

This would result in saturated logits which would lead the row probabilities to converge to one hot vectors:


In [152]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.6]), dim=-1)

tensor([0.1869, 0.1384, 0.2282, 0.1384, 0.3081])

In [ ]:
torch.softmax(torch.tensor([0.1, -0.2, 0.3, -0.2, 0.6]) * 8, dim=-1)

tensor([0.0165, 0.0015, 0.0816, 0.0015, 0.8990])

Observe how the attention probability distribution for the token is basically a one hot vector [0,0,0,0,1].

This means that particular token will aggregate the value from only the last token at initialization!

The scaling scales down the variance of the attention matrix to 1 so that the probabilities generated by softmax are spread apart.


## Residual connections

Residual connections (also called skip connections) are a architectural technique where the input to a layer is added directly to the output of that layer.

### How they work

Mathematically, instead of:

$$\text{output} = F(x)$$

We have:

$$\text{output} = F(x) + x$$

Where $F(x)$ is some transformation (like a neural network layer) and $x$ is the input.

### What they do in GPT

In [Vaswani et al. (2017)](https://arxiv.org/pdf/1706.03762), residual connections appear in two key places within each transformer block:

1. **Around the attention layer**: The input embeddings are added back to the output of the multi-head attention
2. **Around the feedforward layer**: The attention output is added back to the output of the feedforward network

So the data flow looks like:

$$x \rightarrow \text{LayerNorm} \rightarrow \text{Attention} \rightarrow (+x) \rightarrow \text{LayerNorm} \rightarrow \text{FFWD} \rightarrow (+ Attention) \rightarrow \text{next layer}$$

### Why are they important?

**Gradient flow**: Residual connections create direct pathways for gradients to flow backward through the network during training. Without them, gradients can vanish or explode in very deep networks, making training difficult or impossible.

**Identity mapping**: They allow each layer to learn refinements to the representation rather than having to learn the entire transformation from scratch. If a layer doesn't help, the network can essentially learn to ignore it by making F(x) ≈ 0.

**Enables depth**: GPT models have dozens of layers (GPT-3 has 96 layers). Residual connections are crucial for making such deep architectures trainable.

This technique was originally introduced in ResNet for computer vision but has become fundamental to modern transformer architectures.
